# Exploration des donn?es - pr?diction de la consommation ?nerg?tique

Objectif de ce notebook : analyser les donn?es de consommation ?nerg?tique des foyers afin de comprendre les principaux facteurs qui influencent la consommation, comme la taille du foyer, la temp?rature moyenne, la pr?sence de climatisation, la surface du logement, le type de chauffage et le temps pass? ? domicile.

Cette EDA permet aussi de pr?parer les variables utiles pour construire ensuite un mod?le de machine learning supervis? de r?gression. La variable cible du mod?le sera `Energy_Consumption_kWh`, c?est-?-dire la consommation ?nerg?tique du foyer en kWh.

Point de vigilance : le dataset couvre une p?riode courte, du 1er avril 2025 au 8 avril 2025. L?analyse permet donc d??tudier les relations entre les variables sur cette p?riode, mais ne permet pas de conclure sur des tendances longues ou saisonni?res.

## Plan de l'EDA

1. Chargement et verification des datasets
2. Controle qualite : types, valeurs manquantes, doublons...
3. Analyse descriptive des variables principales
4. Analyse des profils clients

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

PROJECT_ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "data" / "household_energy_consumption_enriched.csv").exists()
    ),
    Path.cwd(),
)
DATA_PATH = PROJECT_ROOT / "data" / "household_energy_consumption_enriched.csv"
PLOTS_DIR = PROJECT_ROOT / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

PLOT_PALETTE = ["#2A9D8F", "#E76F51", "#457B9D", "#F4A261", "#8D6A9F", "#90BE6D"]
sns.set_theme(style="whitegrid", context="notebook", palette=PLOT_PALETTE, font_scale=1.05)
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#FBFCFE",
    "axes.edgecolor": "#D7DEE8",
    "axes.labelcolor": "#273142",
    "axes.titlecolor": "#111827",
    "axes.titleweight": "bold",
    "grid.color": "#E7EBF0",
    "grid.linewidth": 0.8,
    "legend.frameon": True,
    "legend.framealpha": 0.95,
    "legend.facecolor": "white",
    "legend.edgecolor": "#E1E6EF",
})


def finalize_plot(fig, filename):
    """Apply shared styling and export the figure to the plots folder."""

    fig.tight_layout()
    fig.savefig(PLOTS_DIR / filename, dpi=160, bbox_inches="tight")
    return fig

## 1. Chargement des donnees

In [ ]:
# Chargement des donn?es enrichies

df = pd.read_csv(DATA_PATH)

datasets_overview = pd.DataFrame({
    "dataset": [DATA_PATH.name],
    "lignes": [len(df)],
    "colonnes": [df.shape[1]],
})

datasets_overview

## 2. Controle qualite

On verifie les types, les valeurs manquantes, les doublons et les incoherences simples avant de produire des graphiques. Cette etape evite d'interpreter des distributions faussees.

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Has_AC"] = df["Has_AC"].astype("string").str.strip().str.title()
df["heating_type"] = df["heating_type"].astype("string").str.strip().str.title()
df["surface_m2"] = pd.to_numeric(df["surface_m2"], errors="coerce")
df["hours_at_home"] = pd.to_numeric(df["hours_at_home"], errors="coerce")
df["Has_AC_Binary"] = df["Has_AC"].map({"Yes": 1, "No": 0})

df["year"] = df["Date"].dt.year
df["month"] = df["Date"].dt.month
df["day_of_week"] = df["Date"].dt.day_name()
df["week"] = df["Date"].dt.isocalendar().week.astype("Int64")

df["consumption_per_person"] = df["Energy_Consumption_kWh"] / df["Household_Size"].replace(0, np.nan)
df["surface_per_person"] = df["surface_m2"] / df["Household_Size"].replace(0, np.nan)
df["peak_usage_ratio"] = df["Peak_Hours_Usage_kWh"] / df["Energy_Consumption_kWh"].replace(0, np.nan)

df.head()

In [ ]:
# Vérification de la qualité des données

quality_report = pd.DataFrame({
    "type": df.dtypes.astype(str),
    "valeurs_manquantes": df.isna().sum(),
    "taux_valeurs_manquantes": df.isna().mean(),
    "valeurs_uniques": df.nunique()
})

quality_report

In [ ]:
# Vérification des doublons et informations générales

duplicate_rows = df.duplicated().sum()
duplicate_household_date = df.duplicated(subset=["Household_ID", "Date"]).sum()

print("Doublons exacts :", duplicate_rows)
print("Doublons Household_ID + Date :", duplicate_household_date)
print("Nombre de foyers distincts :", df["Household_ID"].nunique())
print("Période couverte :", df["Date"].min(), "->", df["Date"].max())

In [ ]:
# V?rification de la coh?rence des donn?es

coherence_checks = pd.Series({
    "consommation_negative_or_zero": (df["Energy_Consumption_kWh"] <= 0).sum(),
    "peak_usage_negative": (df["Peak_Hours_Usage_kWh"] < 0).sum(),
    "peak_usage_greater_than_total": (df["Peak_Hours_Usage_kWh"] > df["Energy_Consumption_kWh"]).sum(),
    "household_size_invalid": (df["Household_Size"] <= 0).sum(),
    "surface_invalid": (df["surface_m2"] <= 0).sum(),
    "hours_at_home_invalid": (~df["hours_at_home"].between(0, 24)).sum(),
    "heating_type_missing": df["heating_type"].isna().sum(),
    "date_missing_or_invalid": df["Date"].isna().sum(),
})

coherence_checks.to_frame(name="nombre")

## 3. Analyse descriptive

Les variables cl?s pour pr?dire la consommation ?nerg?tique sont la taille du foyer, la temp?rature moyenne, la pr?sence de climatisation, les indicateurs d'usage, ainsi que les nouvelles variables g?n?r?es : `surface_m2`, `heating_type` et `hours_at_home`.

In [ ]:
numeric_columns = [
    "Energy_Consumption_kWh",
    "Peak_Hours_Usage_kWh",
    "peak_usage_ratio",
    "Household_Size",
    "surface_m2",
    "surface_per_person",
    "hours_at_home",
    "consumption_per_person",
    "Avg_Temperature_C",
]

df[numeric_columns].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T

In [ ]:
df["Peak_Hours_Usage_kWh"].head(20)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharey=False)

hist_specs = [
    ("Energy_Consumption_kWh", "Distribution de la consommation totale", "Consommation totale (kWh)", PLOT_PALETTE[0]),
    ("Peak_Hours_Usage_kWh", "Distribution de la consommation en heures de pointe", "Heures de pointe (kWh)", PLOT_PALETTE[1]),
    ("consumption_per_person", "Consommation par personne", "kWh par personne", PLOT_PALETTE[2]),
    ("peak_usage_ratio", "Part de la consommation en heures de pointe", "Ratio heures de pointe", PLOT_PALETTE[3]),
]

for ax, (column, title, xlabel, color) in zip(axes.flat, hist_specs):
    values = df[column].clip(0, 1.5) if column == "peak_usage_ratio" else df[column]
    sns.histplot(values, bins=35, kde=True, color=color, edgecolor="white", linewidth=0.5, ax=ax)
    ax.set_title(title, pad=12)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Nombre d'observations")

finalize_plot(fig, "distributions_consommation.png")
plt.show()

### Variables logement g?n?r?es

Les nouvelles colonnes enrichissent le profil du foyer sans remplacer les variables existantes : la surface est stable par foyer, le type de chauffage est cat?goriel et les heures ? domicile varient par jour.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

sns.histplot(df["surface_m2"], bins=35, kde=True, color=PLOT_PALETTE[0], ax=axes[0])
axes[0].set_title("Distribution des surfaces")
axes[0].set_xlabel("Surface (m2)")

sns.countplot(data=df, x="heating_type", order=df["heating_type"].value_counts().index, palette="crest", ax=axes[1])
axes[1].set_title("R?partition des types de chauffage")
axes[1].set_xlabel("Type de chauffage")
axes[1].tick_params(axis="x", rotation=25)

sns.histplot(df["hours_at_home"], bins=28, kde=True, color=PLOT_PALETTE[2], ax=axes[2])
axes[2].set_title("Temps quotidien ? domicile")
axes[2].set_xlabel("Heures ? domicile")

finalize_plot(fig, "features_logement_generees.png")
plt.show()

### Surface du foyer et consommation ?nerg?tique

Cette visualisation v?rifie si les logements plus grands sont associ?s ? une consommation plus ?lev?e. Le nuage de points donne la relation observation par observation, tandis que la moyenne par tranche de surface rend la tendance plus lisible.

In [ ]:
surface_sample = df.sample(min(12000, len(df)), random_state=42)
surface_bins = [0, 50, 75, 100, 125, 150, 200]
surface_labels = ["<=50", "51-75", "76-100", "101-125", "126-150", ">150"]
surface_summary = (
    df.assign(surface_segment=pd.cut(df["surface_m2"], bins=surface_bins, labels=surface_labels))
    .groupby("surface_segment", observed=True)["Energy_Consumption_kWh"]
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))

sns.scatterplot(
    data=surface_sample,
    x="surface_m2",
    y="Energy_Consumption_kWh",
    hue="Household_Size",
    palette="viridis",
    alpha=0.35,
    s=18,
    linewidth=0,
    ax=axes[0],
)
axes[0].set_title("Consommation selon la surface")
axes[0].set_xlabel("Surface du logement (m2)")
axes[0].set_ylabel("Consommation totale (kWh)")
axes[0].legend(title="Taille foyer", frameon=True)

sns.barplot(
    data=surface_summary,
    x="surface_segment",
    y="Energy_Consumption_kWh",
    color=PLOT_PALETTE[0],
    ax=axes[1],
)
axes[1].set_title("Consommation moyenne par tranche de surface")
axes[1].set_xlabel("Surface du logement (m2)")
axes[1].set_ylabel("Consommation moyenne (kWh)")

finalize_plot(fig, "surface_consommation_energie.png")
plt.show()

### Consommation ?nerg?tique selon le type de chauffage

Cette visualisation compare la consommation en kWh selon le chauffage principal. Elle permet de rep?rer les diff?rences de niveau et de dispersion entre les profils de logement.

In [ ]:
heating_order = df.groupby("heating_type")["Energy_Consumption_kWh"].median().sort_values().index
heating_summary = (
    df.groupby("heating_type", observed=True)["Energy_Consumption_kWh"]
    .agg(consommation_moyenne="mean", consommation_mediane="median")
    .reindex(heating_order)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))

sns.boxplot(
    data=df,
    x="heating_type",
    y="Energy_Consumption_kWh",
    order=heating_order,
    palette="crest",
    linewidth=1.1,
    fliersize=1.8,
    ax=axes[0],
)
axes[0].set_title("Distribution de l'?nergie par chauffage")
axes[0].set_xlabel("Type de chauffage")
axes[0].set_ylabel("Consommation totale (kWh)")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(
    data=heating_summary,
    x="heating_type",
    y="consommation_moyenne",
    palette="mako",
    ax=axes[1],
)
axes[1].set_title("Consommation moyenne par chauffage")
axes[1].set_xlabel("Type de chauffage")
axes[1].set_ylabel("Consommation moyenne (kWh)")
axes[1].tick_params(axis="x", rotation=20)

finalize_plot(fig, "energie_type_chauffage.png")
plt.show()

Interprétation des distributions

Le premier graphique montre la distribution de la consommation énergétique totale. Les valeurs sont réparties entre environ 0 et 20 kWh, avec une forte concentration proche de 20 kWh. Cela indique que plusieurs foyers atteignent des niveaux élevés de consommation.

Le deuxième graphique montre la distribution de la consommation en heures de pointe. La plupart des valeurs se situent entre 1 et 6 kWh, avec quelques consommations plus élevées autour de 8 à 10 kWh. Cela montre que les heures de pointe représentent une part importante de la consommation.

Le troisième graphique montre la consommation par personne. La majorité des foyers se situe autour de 2 à 4 kWh par personne. Cela permet de comparer les foyers de tailles différentes de manière plus juste.

Le quatrième graphique montre la part de la consommation réalisée en heures de pointe. La plupart des valeurs sont comprises entre 0,30 et 0,50, ce qui signifie que les heures de pointe représentent souvent entre 30 % et 50 % de la consommation totale.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(
    data=df,
    x="Household_Size",
    y="Energy_Consumption_kWh",
    palette="crest",
    linewidth=1.1,
    fliersize=2.5,
    ax=axes[0],
)
axes[0].set_title("Consommation selon la taille du foyer", pad=12)
axes[0].set_xlabel("Taille du foyer")
axes[0].set_ylabel("Consommation totale (kWh)")

sns.boxplot(
    data=df,
    x="Has_AC",
    y="Energy_Consumption_kWh",
    palette={"No": PLOT_PALETTE[2], "Yes": PLOT_PALETTE[1]},
    linewidth=1.1,
    fliersize=2.5,
    ax=axes[1],
)
axes[1].set_title("Consommation selon la presence de climatisation", pad=12)
axes[1].set_xlabel("Climatisation")
axes[1].set_ylabel("Consommation totale (kWh)")

finalize_plot(fig, "consommation_taille_climatisation.png")
plt.show()

À gauche : La consommation énergétique augmente avec la taille du foyer. Cela semble logique, car un foyer avec plus de personnes utilise généralement plus d’électricité.

À droite : Les foyers disposant d’une climatisation ont une consommation énergétique plus élevée que ceux qui n’en ont pas. Cela montre que la présence de climatisation est un facteur important dans la consommation d’énergie.